# D18 · Agentes y proyecto integrador de IA aplicada

**Formación Pública — Bootcamp de Ciencia de Datos para funcionarios públicos**
Bloque avanzado · IA aplicada (opcional) · Semana 21 · **Módulo de cierre**

---

## ¿Qué vas a lograr?

Llegaste al final del bloque avanzado. Tienes un arsenal: sabes **pronosticar** (D14), **clasificar** texto (D15/D16), **extraer** datos (D16) y **responder con tus documentos** (D17). Pero hasta ahora *tú* decidías qué técnica usar en cada caso. Hoy construyes algo que decide **solo**: un **agente**.

Un agente de IA recibe una petición en lenguaje natural, **decide qué herramienta usar** para resolverla, la ejecuta y te entrega la respuesta. Es el patrón que está detrás de los asistentes que "hacen cosas", no solo "conversan". Y como cierre del bootcamp, lo armarás **integrando todo lo que ya construiste**, sobre datos reales del Estado.

Al terminar serás capaz de:

- Entender qué es un agente y su ciclo **decidir → actuar → responder**.
- Empaquetar capacidades como **herramientas** que el agente puede invocar.
- Construir un **enrutador** que elige la herramienta correcta para cada petición.
- Hacer que el agente sea **transparente** (diga qué usó) y **seguro** (proteja datos personales).

**Competencia de salida:** orquestar varias capacidades de IA en un único asistente que resuelve tareas reales de gobierno de principio a fin.

**Entregable:** que las **4 celdas de chequeo** muestren ✅.


## Tabla de Datos Reales

Este módulo integra tres tipos de datos reales de la administración pública de Chile que tu agente utilizará para resolver consultas:

1. **Catálogo de Rubros de MercadoPúblico (Simplificado):**
   - Alimentos, bebidas y tabaco
   - Equipamiento y suministros médicos
   - Equipos, accesorios y suministros de oficina
   - Maquinaria para construcción y edificación
   - Medicamentos y productos farmacéuticos
   - Muebles y mobiliario
   - Servicios de construcción y mantenimiento
   - Servicios de limpieza industrial

2. **Bases de Licitación 2239-8-LR25 (RAG):**
   - Criterios de evaluación (precio 75%, experiencia 20%, etc.).
   - Garantías de fiel cumplimiento ($400.000).
   - Duración del contrato (36 meses).
   - Ausencia de exigencia de seriedad de la oferta.

3. **Gasto Público Histórico Anual (2019-2025):**
   
| Año | Gasto Anual Registrado (CLP) |
|---|---|
| 2019 | 8.141.600.896.708 |
| 2020 | 8.945.139.620.728 |
| 2021 | 9.835.060.128.510 |
| 2022 | 11.494.882.965.920 |
| 2023 | 12.483.986.700.171 |
| 2024 | 15.601.131.083.597 |
| 2025 | 18.430.546.563.769 |

*Fuente de datos: ChileCompra / MercadoPúblico / Dirección de Compras y Contratación Pública (ChileCompra).*


## 1. ¿Qué es un agente?

Un programa normal sigue una receta fija: haz A, luego B, luego C. Un **agente** es distinto: ante una petición, **primero decide qué hacer** y recién entonces actúa. Su ciclo básico es:

1. **Percibir:** recibe la petición del usuario.
2. **Decidir:** elige, de entre sus **herramientas**, la más adecuada.
3. **Actuar:** ejecuta esa herramienta.
4. **Responder:** entrega el resultado (idealmente diciendo *qué* hizo).

La pieza nueva es la **decisión**. Lo demás —las herramientas— ya lo construiste en los módulos anteriores.

### Herramientas

Una **herramienta** es simplemente una función que hace una cosa bien. Tu agente tendrá estas, heredadas del bootcamp:

| Herramienta | Qué hace | Viene de |
|---|---|---|
| `clasificar_compra` | dice a qué rubro pertenece una compra | D15 / D16 |
| `consultar_ficha` | responde preguntas sobre una licitación (RAG) | D17 |
| `pronosticar_gasto` | proyecta el gasto público de un año | D14 |
| `responder_general` | responde con el LLM cuando nada encaja | D16 |

### ¿Quién decide? El enrutador

La decisión la toma un **enrutador** (router): mira la petición y elige la herramienta. En producción, esa decisión suele tomarla **un LLM** (justo lo que viste en D16: darle opciones y pedirle que elija). Aquí lo haremos con **reglas simples** (palabras clave), por una buena razón: así el cuaderno es **100% reproducible y verificable sin conexión**. El patrón es idéntico; solo cambia el "cerebro" que decide.

### Transparencia y seguridad: no son opcionales

Un agente en el Estado debe ser **transparente** (decir qué herramienta usó, para poder auditarlo) y **seguro** (no filtrar datos personales a servicios externos). Cerraremos con ambas cosas: una **traza** de la decisión y el **guardrail** de privacidad que construiste en D16.


## 2. Preparación: tus herramientas, ya listas

Para que te concentres en el **agente**, dejamos provistas las herramientas (versiones compactas de lo que ya hiciste) y la conexión al LLM con el patrón "en vivo o caché". Ejecuta la celda y revisa qué hace cada función.


In [ ]:
# Instala el SDK oficial de Google (en Colab; si ya está, no hace nada)
!pip install -q google-genai

import os, re, unicodedata, urllib.request
import pandas as pd

MODELO = "gemini-2.5-flash"

# ---------- utilidades de texto (de D15) ----------
STOPWORDS = {"de","y","e","o","u","para","el","la","los","las","un","una","en","con",
             "a","del","al","por","su","sus","se","que","es","son"}
def _sa(s):
    s = unicodedata.normalize("NFKD", str(s).lower())
    return "".join(c for c in s if not unicodedata.combining(c))
def _tok(texto):
    base = re.sub(r"[^a-z0-9\s]", " ", _sa(texto))
    return [t for t in base.split() if t not in STOPWORDS and len(t) >= 3]

# ---------- datos reales ----------
# Serie anual de gasto público (D14), en CLP — años completos 2019–2025
# Si el archivo no existe localmente (ej: en Colab), intentamos descargarlo desde GitHub
if not os.path.exists("gasto_anual.csv"):
    try:
        url = "https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/C3-agentes/gasto_anual.csv"  # Reemplazar por la URL real del repositorio al publicar
        urllib.request.urlretrieve(url, "gasto_anual.csv")
        print("gasto_anual.csv descargado automáticamente.")
    except Exception:
        print("Aviso: No se pudo descargar gasto_anual.csv automáticamente.")

df_gasto = pd.read_csv("gasto_anual.csv")
GASTO = dict(zip(df_gasto["anio"], df_gasto["gasto"]))

# Rubros reales (subconjunto del catálogo, D15/D16)
RUBROS = ["Alimentos, bebidas y tabaco", "Equipamiento y suministros médicos",
          "Equipos, accesorios y suministros de oficina", "Maquinaria para construcción y edificación",
          "Medicamentos y productos farmacéuticos", "Muebles y mobiliario",
          "Servicios de construcción y mantenimiento", "Servicios de limpieza industrial"]

# Base de conocimiento de la ficha real 2239-8-LR25 (D17)
KB = [
    "La evaluación de las ofertas pondera: precio 75%, experiencia 20%, programa de integridad 3% y requisitos formales 2%.",
    "La garantía de fiel cumplimiento del contrato es por 400.000 pesos, dentro de 20 días hábiles desde la adjudicación.",
    "La duración del contrato es de 36 meses.",
    "Esta licitación no exige garantía de seriedad de la oferta.",
]

# ---------- LLM: en vivo o caché (D16) ----------
def _api_key():
    try:
        from google.colab import userdata
        k = userdata.get("GEMINI_API_KEY")
        if k: return k
    except Exception:
        pass
    return os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
API_KEY = _api_key()
def preguntar_llm(prompt, tarea="general"):
    if API_KEY:
        try:
            from google import genai
            client = genai.Client(api_key=API_KEY)
            return (client.models.generate_content(model=MODELO, contents=prompt).text or "")
        except Exception as e:
            print(f"⚠️ La llamada en vivo falló ({e}). Uso la respuesta en caché.")
    return "No tengo una herramienta específica para eso; te respondo de forma general."

# ---------- HERRAMIENTAS provistas ----------
def clasificar_compra(texto):
    q = set(_tok(texto))
    return max(RUBROS, key=lambda r: len(q & set(_tok(r))))

def consultar_ficha(pregunta):
    q = set(_tok(pregunta))
    mejor = max(KB, key=lambda t: len(q & set(_tok(t))))
    return mejor if len(q & set(_tok(mejor))) > 0 else "No está en la ficha."

def responder_general(pregunta):
    return preguntar_llm(pregunta, tarea="general")

def es_seguro_enviar(texto):
    return re.search(r"\b\d{1,2}\.?\d{3}\.?\d{3}-[\dkK]\b", texto) is None  # detecta RUT

def _extraer_anio(texto):
    m = re.search(r"\b(20\d{2})\b", texto)
    return int(m.group(1)) if m else None

print("Herramientas listas:", [f.__name__ for f in
      (clasificar_compra, consultar_ficha, responder_general)])
print("Modo LLM:", "EN VIVO" if API_KEY else "CACHÉ (offline)")


## Manejo de Errores Comunes

Al diseñar y desplegar agentes integradores de IA en el sector público, es frecuente encontrarse con los siguientes errores. Aprende a identificarlos y solucionarlos:

1. **`AttributeError: 'NoneType' object has no attribute 'group'` en el pronóstico:**
   Ocurre si intentas leer el año usando `_extraer_anio` y la petición no contiene ningún año de 4 dígitos (lo que hace que `re.search` devuelva `None`). En la lógica de tu `agente`, siempre debes verificar si se pudo extraer el año, y de lo contrario, definir un año por defecto (como 2026):
   `anio = _extraer_anio(peticion) or 2026`

2. **Petición redirigida a la herramienta incorrecta (Enrutamiento fallido):**
   Si el enrutador manda una consulta de compras a la ficha de licitación o viceversa, revisa el orden de tus condicionales en `elegir_herramienta`. Recuerda que las reglas se evalúan de forma secuencial y excluyente. Asegúrate de estructurar bien las palabras clave de búsqueda.

3. **Pérdida de traza de auditoría en la respuesta:**
   Un error común en la administración del Estado es entregar una respuesta automatizada sin registrar qué herramienta o algoritmo se utilizó. Al diseñar `agente_seguro`, es obligatorio devolver un diccionario con la estructura `{"herramienta": ..., "respuesta": ...}`. Esto permite registrar la traza y depurar o auditar las decisiones del agente.

4. **Fuga de información confidencial en el envío al LLM:**
   Si tu agente procesa textos directamente sin validar primero la presencia de datos personales (como un RUT), podrías vulnerar normativas de privacidad. Tu función `agente_seguro` debe llamar siempre a `es_seguro_enviar` como primer filtro obligatorio, bloqueando y devolviendo un mensaje informativo si se detectan datos sensibles.


## Ejercicio 1 — Construye una herramienta: pronosticar el gasto

Tu agente necesita poder proyectar el gasto futuro. Reutilizarás la lógica de **D14** (crecimiento medio) para crear la herramienta `pronosticar_gasto(anio)`.

Programa `pronosticar_gasto(anio)` que:

1. Tome los valores de `GASTO` ordenados por año.
2. Calcule el **crecimiento medio** año a año.
3. Parta del último año conocido (2025) y aplique ese crecimiento, año por año, hasta llegar a `anio`.
4. Devuelva el monto proyectado (un número).

> 💡 **Pista:**
> ```python
> anios = sorted(GASTO)
> valores = [GASTO[a] for a in anios]
> crecimientos = [valores[i] / valores[i-1] - 1 for i in range(1, len(valores))]
> g = sum(crecimientos) / len(crecimientos)
> valor = GASTO[anios[-1]]
> for _ in range(anio - anios[-1]):
>     valor *= (1 + g)
> return valor
> ```

> ⚠️ **Recuerda de D14:** un pronóstico es una estimación con supuestos. Aquí suponemos que el crecimiento histórico continúa; sirve como referencia, no como certeza.


In [ ]:
def pronosticar_gasto(anio):
    # TODO: crecimiento medio desde 2025 hasta `anio`; devuelve el monto proyectado
    return None   # <-- reemplaza por tu implementación

res = pronosticar_gasto(2026)
print(f"{res:,.0f} CLP" if res is not None else "todavía vacío")


In [ ]:
# Celda de chequeo — Ejercicio 1
def _chequeo_1():
    anios = sorted(GASTO); vals = [GASTO[a] for a in anios]
    crec = [vals[i]/vals[i-1]-1 for i in range(1, len(vals))]
    g = sum(crec)/len(crec)
    def esperado(anio):
        v = GASTO[anios[-1]]
        for _ in range(anio - anios[-1]): v *= (1+g)
        return v
    try:
        f = pronosticar_gasto
    except NameError:
        print("❌ Aún no defines `pronosticar_gasto`."); return
    try:
        r = f(2026)
        assert r is not None, "Devuelve None. Completa la implementación."
        assert abs(float(r) - esperado(2026)) < 1, "El pronóstico de 2026 no coincide con el crecimiento medio."
        assert abs(float(f(2027)) - esperado(2027)) < 1, "Revisa que apliques el crecimiento año a año hasta `anio`."
        print("✅ Correcto: herramienta de pronóstico lista.")
        print(f"   Gasto proyectado 2026: {f(2026):,.0f} CLP   ·   2027: {f(2027):,.0f} CLP")
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")
    except Exception as e:
        print(f"❌ La función falló: {e}.")

_chequeo_1()

## Ejercicio 2 — El enrutador: ¿qué herramienta uso?

El cerebro del agente. Programa `elegir_herramienta(peticion)` que devuelva **el nombre** de la herramienta adecuada según palabras clave. Debe devolver uno de: `"clasificar"`, `"pronostico"`, `"ficha"` o `"general"`.

Reglas (en este orden):

1. Si la petición habla de **comprar/adquirir/rubro** → `"clasificar"`.
2. Si habla de **pronóstico/proyección/gastar/futuro** → `"pronostico"`.
3. Si habla de la **licitación/ficha/garantía/plazo/evaluación/contrato/precio** → `"ficha"`.
4. Si nada calza → `"general"`.

> 💡 **Pista:** normaliza con `_sa(peticion)` y usa `any(palabra in texto for palabra in [...])` para cada grupo. Respeta el orden de las reglas.

> ⚠️ **Esto es la "decisión" del agente.** En producción la tomaría un LLM (como en D16); aquí, reglas simples para que sea reproducible. El patrón —elegir entre herramientas— es el mismo.


In [ ]:
def elegir_herramienta(peticion):
    # TODO: devuelve "clasificar" / "pronostico" / "ficha" / "general" según palabras clave
    return None   # <-- reemplaza por tu implementación

for p in ["Quiero comprar muebles para la oficina",
          "¿Cuánto gastará el Estado el 2027?",
          "¿Qué pondera el precio en la evaluación?",
          "Hola, ¿cómo estás?"]:
    print(f"{elegir_herramienta(p)!s:12s} ← {p}")


In [ ]:
# Celda de chequeo — Ejercicio 2
def _chequeo_2():
    try:
        f = elegir_herramienta
    except NameError:
        print("❌ Aún no defines `elegir_herramienta`."); return
    casos = {
        "Quiero comprar muebles para la oficina": "clasificar",
        "¿Cuánto gastará el Estado el 2027?": "pronostico",
        "¿Qué pondera el precio en la evaluación de la licitación?": "ficha",
        "Hola, ¿cómo estás?": "general",
    }
    try:
        for peticion, esperado in casos.items():
            obtenido = f(peticion)
            assert obtenido == esperado, f"«{peticion}» debería ir a «{esperado}», no «{obtenido}»."
        print("✅ Correcto: el enrutador elige bien la herramienta para cada petición.")
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")
    except Exception as e:
        print(f"❌ La función falló: {e}.")

_chequeo_2()

## Ejercicio 3 — El agente: decidir y actuar

Ahora unimos enrutador + herramientas. Programa `agente(peticion)` que:

1. Use `elegir_herramienta(peticion)` para decidir.
2. **Despache** a la herramienta correcta:
   - `"clasificar"` → `clasificar_compra(peticion)`
   - `"ficha"` → `consultar_ficha(peticion)`
   - `"pronostico"` → busca el año con `_extraer_anio(peticion)` (si no hay, usa 2026) y devuelve un texto como `f"Gasto proyectado para {anio}: {pronosticar_gasto(anio):,.0f} CLP"`
   - `"general"` → `responder_general(peticion)`
3. Devuelva la respuesta.

> 💡 **Pista:** un `if/elif` por cada herramienta. Para el pronóstico:
> `anio = _extraer_anio(peticion) or 2026`.

> ⚠️ **Error típico:** olvidar el caso `"general"`. Un agente robusto siempre tiene una salida por defecto.


In [ ]:
def agente(peticion):
    # TODO: decide con elegir_herramienta y despacha a la herramienta correcta
    return None   # <-- reemplaza por tu implementación

print(agente("Quiero comprar muebles y mobiliario para la oficina"))
print(agente("¿Cuánto gastará el Estado el 2027?"))
print(agente("¿Qué pondera el precio en la evaluación?"))


In [ ]:
# Celda de chequeo — Ejercicio 3
def _chequeo_3():
    try:
        f = agente
    except NameError:
        print("❌ Aún no defines `agente`."); return
    try:
        r1 = f("Quiero comprar muebles y mobiliario para la oficina")
        assert r1 is not None, "Devuelve None. Completa la implementación."
        assert "muebles" in _sa(str(r1)), "Una compra de muebles debería clasificarse en el rubro de muebles."
        r2 = f("¿Cuánto gastará el Estado el 2027?")
        assert "2027" in str(r2) and "clp" in _sa(str(r2)), "El pronóstico debería mencionar el año 2027 y el monto en CLP."
        r3 = f("¿Qué pondera el precio en la evaluación?")
        assert "75" in str(r3), "La consulta a la ficha debería traer el pasaje de la evaluación (precio 75%)."
        print("✅ Correcto: el agente decide y actúa con la herramienta adecuada.")
        print("   compra →", r1)
        print("   pronóstico →", r2)
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")
    except Exception as e:
        print(f"❌ La función falló: {e}.")

_chequeo_3()

## Ejercicio 4 — Cierre: un agente transparente y seguro

Último paso del bootcamp. Un agente público debe poder **auditarse** (decir qué hizo) y **no filtrar datos personales**. Programa `agente_seguro(peticion)` que devuelva un **diccionario** con la traza de la decisión, y que **bloquee** peticiones con datos personales.

Tu función debe:

1. Si la petición **no es segura** (`es_seguro_enviar(peticion)` es `False`), devolver:
   `{"herramienta": "bloqueado", "respuesta": "Petición bloqueada: contiene datos personales (RUT)."}`
2. Si es segura, calcular la herramienta con `elegir_herramienta`, ejecutar `agente(peticion)`, y devolver:
   `{"herramienta": <nombre>, "respuesta": <resultado>}`

> 💡 **Pista:** primero el guardrail (`if not es_seguro_enviar(peticion): ...`), después la ejecución normal.

> ⚠️ **Por qué importa:** este diccionario es tu **traza de auditoría**. En el Estado, poder responder "¿por qué el sistema hizo esto?" no es un lujo: es un requisito.


In [ ]:
def agente_seguro(peticion):
    # TODO: bloquea si hay datos personales; si no, ejecuta y devuelve {herramienta, respuesta}
    return None   # <-- reemplaza por tu implementación

print(agente_seguro("Quiero comprar muebles y mobiliario para la oficina"))
print(agente_seguro("El proveedor RUT 12.345.678-9 entrega el martes"))


In [ ]:
# Celda de chequeo — Ejercicio 4
def _chequeo_4():
    try:
        f = agente_seguro
    except NameError:
        print("❌ Aún no defines `agente_seguro`."); return
    try:
        ok = f("Quiero comprar muebles y mobiliario para la oficina")
        assert isinstance(ok, dict) and "herramienta" in ok and "respuesta" in ok, \
            "Debe devolver un diccionario con 'herramienta' y 'respuesta'."
        assert ok["herramienta"] == "clasificar", "Esa petición debería usar la herramienta 'clasificar'."
        assert "muebles" in _sa(str(ok["respuesta"])), "La respuesta debería traer el rubro de muebles."
        bloq = f("El proveedor RUT 12.345.678-9 entrega el martes")
        assert bloq["herramienta"] == "bloqueado", "Una petición con RUT debe quedar BLOQUEADA."
        print("✅ Correcto: agente transparente (traza) y seguro (bloquea datos personales).")
        print("   Traza:", ok)
        print("   Bloqueo:", bloq)
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")
    except Exception as e:
        print(f"❌ La función falló: {e}.")

_chequeo_4()

## Cierre — Lo que te llevas (y el fin del bloque avanzado)

- Un **agente** no sigue una receta fija: **decide** qué herramienta usar, **actúa** y **responde**.
- Las **herramientas** son tus capacidades previas empaquetadas: pronosticar (D14), clasificar (D15/D16), consultar documentos (D17), responder (D16).
- El **enrutador** es la decisión; puede ser por reglas (reproducible) o por LLM (flexible). El patrón es el mismo.
- Un agente público se construye con **transparencia** (traza auditable) y **seguridad** (guardrails) desde el primer día, no después.

### Lo que lograste en todo el bloque avanzado
Tomaste datos reales del Estado y, paso a paso, construiste series temporales, un primer NLP, llamadas a un LLM, un sistema RAG y, hoy, un agente que orquesta todo. Eso es **IA aplicada al servicio público**, hecha por ti, con datos reales y de forma responsable.

### Hacia el Proyecto Final
Con esto cierras D14–D18. El siguiente hito es el **Proyecto Final**: tomar un problema real de tu institución y resolverlo de punta a punta con lo aprendido. Nos vemos ahí.

---
*Formación Pública · Contenido bajo licencia CC BY 4.0. Datos: ChileCompra / MercadoPúblico. LLM: Google Gemini (capa gratuita).*
